# ASTA Moondream2 LoRA Fine-Tuning Notebook

This notebook trains a lightweight VLM classifier for the 12-class seasonal color task.

Expected Google Drive structure:

```text
MyDrive/ASTA/
└── data/
    ├── moondream_train_classification.jsonl
    ├── moondream_test_classification.jsonl
    └── training_crops/
        ├── train/
        └── test/
```

The model learns only this output:

```json
{"season":"Autumn","subtype":"Deep"}
```

The app should still use templates in `analyzer.py` for palette, outfit colors, avoid colors, makeup tips, and summary.

In [1]:
# ============================================================
# 1. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/ASTA"

TRAIN_JSONL = f"{PROJECT_ROOT}/data/moondream_train_classification.jsonl"
TEST_JSONL  = f"{PROJECT_ROOT}/data/moondream_test_classification.jsonl"

CHECKPOINT_DIR = f"{PROJECT_ROOT}/checkpoints/moondream2_lora"
SAVE_DIR       = f"{PROJECT_ROOT}/moondream_season_lora"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_JSONL:", TRAIN_JSONL)
print("TEST_JSONL:", TEST_JSONL)
print("SAVE_DIR:", SAVE_DIR)

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/ASTA
TRAIN_JSONL: /content/drive/MyDrive/ASTA/data/moondream_train_classification.jsonl
TEST_JSONL: /content/drive/MyDrive/ASTA/data/moondream_test_classification.jsonl
SAVE_DIR: /content/drive/MyDrive/ASTA/moondream_season_lora


In [2]:
# ============================================================
# 2. Install dependencies
# ============================================================

!pip -q install "transformers==4.41.2" "peft==0.8.2" "accelerate>=0.30.0" "safetensors" "einops" "pillow" "tqdm" "pandas" "scikit-learn"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 75.1 MB/s eta 0:00:00


In [3]:
# ============================================================
# 3. Verify GPU
# ============================================================

import torch

assert torch.cuda.is_available(), "No GPU detected. In Colab, go to Runtime > Change runtime type > GPU."

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("Torch:", torch.__version__)
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

GPU: Tesla T4
CUDA: 12.8
Torch: 2.10.0+cu128
VRAM GB: 15.64


In [4]:
# ============================================================
# 4. Imports and config
# ============================================================

import os
import re
import json
import random
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

from sklearn.metrics import confusion_matrix, classification_report

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Base model config
# ------------------------------------------------------------
MODEL_ID = "vikhyatk/moondream2"
REVISION = "2024-08-26"

# ------------------------------------------------------------
# Training config
# ------------------------------------------------------------
NUM_EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 1e-4
MAX_TEXT_TOKENS = 96

DEVICE = "cuda"

# ------------------------------------------------------------
# 4-season classification labels
# ------------------------------------------------------------
CLASS_NAMES = [
    "Autumn",
    "Spring",
    "Summer",
    "Winter",
]

VALID_SEASONS = {"Spring", "Summer", "Autumn", "Winter"}

# Keep this for compatibility if older parser/evaluation cells still reference it.
VALID_SUBTYPES = set()

# ------------------------------------------------------------
# Single source of truth prompt
# This must match:
# 1. generate_dataset.py TRAINING_PROMPT
# 2. analyzer.py PROMPT
# 3. notebook evaluation PROMPT
# ------------------------------------------------------------
MASTER_PROMPT = """
Classify this face into exactly one of these 4 seasonal color classes:

Autumn, Spring, Summer, Winter.

Return ONLY this JSON format:
{"season":"Winter"}

No explanation. No markdown. No extra words.
""".strip()

PROMPT = MASTER_PROMPT

print("Config loaded.")
print(f"MASTER_PROMPT ({len(MASTER_PROMPT)} chars):")
print(MASTER_PROMPT)

Config loaded.
MASTER_PROMPT (200 chars):
Classify this face into exactly one of these 4 seasonal color classes:

Autumn, Spring, Summer, Winter.

Return ONLY this JSON format:
{"season":"Winter"}

No explanation. No markdown. No extra words.


In [5]:
# ============================================================
# 5. Dataset loader and path check
# ============================================================

class ASTAJsonlDataset(Dataset):
    def __init__(self, jsonl_path, project_root, limit=None):
        self.jsonl_path = Path(jsonl_path)
        self.project_root = Path(project_root)
        self.samples = []

        assert self.jsonl_path.exists(), f"JSONL not found: {self.jsonl_path}"

        with open(self.jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                sample = json.loads(line)
                rel_path = sample["image_path"].replace("\\", "/")
                abs_path = self.project_root / rel_path

                qa = sample["qa"][0]
                answer = qa["answer"]

                try:
                    parsed = json.loads(answer)
                    label = parsed["season"]
                except Exception:
                    label = "PARSE_ERROR"

                self.samples.append({
                    "image_path": str(abs_path),
                    "relative_image_path": rel_path,
                    "question": qa["question"],
                    "answer": answer,
                    "label": label,
                })

        if limit is not None:
            self.samples = self.samples[:limit]

        missing = [s for s in self.samples if not Path(s["image_path"]).exists()]

        if missing:
            print(f"WARNING: Missing {len(missing)} images. These will be skipped.")
            print("First missing:", missing[0]["image_path"])

            self.samples = [
                s for s in self.samples
                if Path(s["image_path"]).exists()
            ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item["image_path"]).convert("RGB")
        return {
            "image": image,
            "question": item["question"],
            "answer": item["answer"],
            "label": item["label"],
            "image_path": item["image_path"],
        }


train_ds = ASTAJsonlDataset(TRAIN_JSONL, PROJECT_ROOT)
test_ds  = ASTAJsonlDataset(TEST_JSONL, PROJECT_ROOT)

print("Train samples:", len(train_ds))
print("Test samples:", len(test_ds))

print("\nTrain label counts:")
print(pd.Series([s["label"] for s in train_ds.samples]).value_counts().sort_index())

print("\nTest label counts:")
print(pd.Series([s["label"] for s in test_ds.samples]).value_counts().sort_index())

print("\nFirst train sample:")
print(train_ds.samples[0])

First missing: /content/drive/MyDrive/ASTA/data/training_crops/train/winter/train_winter_Megan_Fox_1_0.png
Train samples: 4000
Test samples: 912

Train label counts:
Autumn    1044
Spring     978
Summer     943
Winter    1035
Name: count, dtype: int64

Test label counts:
Autumn    259
Spring    203
Summer    186
Winter    264
Name: count, dtype: int64

First train sample:
{'image_path': '/content/drive/MyDrive/ASTA/data/training_crops/train/winter/train_winter_gwen_stefani3.png', 'relative_image_path': 'data/training_crops/train/winter/train_winter_gwen_stefani3.png', 'question': 'Classify this face into exactly one of these 4 seasonal color classes:\n\nAutumn, Spring, Summer, Winter.\n\nReturn ONLY this JSON format:\n{"season":"Winter"}\n\nNo explanation. No markdown. No extra words.', 'answer': '{"season":"Winter"}', 'label': 'Winter'}


In [6]:
# ============================================================
# 6. Load Moondream2
# ============================================================

torch.cuda.empty_cache()

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map={"": DEVICE},
)

model.eval()
print("Loaded:", MODEL_ID, REVISION)
print("Model dtype:", next(model.parameters()).dtype)

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

vision_encoder.py: 0.00B [00:00, ?B/s]

modeling_phi.py: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.74G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loaded: vikhyatk/moondream2 2024-08-26
Model dtype: torch.float16


In [7]:
# ============================================================
# 7. Attach LoRA to the text model
# ============================================================
# We train only LoRA adapters on the language side.
# The image encoder stays frozen.

for p in model.parameters():
    p.requires_grad = False

# These names work for common transformer attention projections.
target_modules = ["q_proj", "v_proj"]

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

try:
    model.text_model = get_peft_model(model.text_model, lora_config)
    print("LoRA attached to model.text_model with:", target_modules)
except Exception as e:
    print("Initial LoRA target failed:", repr(e))
    print("Scanning linear module names...")

    linear_names = []
    for name, module in model.text_model.named_modules():
        if isinstance(module, nn.Linear):
            linear_names.append(name.split(".")[-1])

    common = sorted(set(linear_names))
    print("Available Linear leaf names:", common[:100])

    fallback_targets = [n for n in ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] if n in common]
    if not fallback_targets:
        fallback_targets = common[-4:]

    print("Fallback target modules:", fallback_targets)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=fallback_targets,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model.text_model = get_peft_model(model.text_model, lora_config)
    print("LoRA attached using fallback targets.")

model.text_model.print_trainable_parameters()

Initial LoRA target failed: ValueError("Target modules {'v_proj', 'q_proj'} not found in the base model. Please check the target modules and try again.")
Scanning linear module names...
Available Linear leaf names: ['Wqkv', 'fc1', 'fc2', 'linear', 'out_proj']
Fallback target modules: ['fc1', 'fc2', 'linear', 'out_proj']
LoRA attached using fallback targets.
trainable params: 5,144,576 || all params: 1,423,415,296 || trainable%: 0.36142480795710097


In [8]:
# ============================================================
# 7.5 Fix FP16 gradient issue for LoRA trainable parameters
# ============================================================

for name, param in model.text_model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

trainable_dtypes = {}

for name, param in model.text_model.named_parameters():
    if param.requires_grad:
        dtype_name = str(param.dtype)
        trainable_dtypes[dtype_name] = trainable_dtypes.get(dtype_name, 0) + param.numel()

print("Trainable parameter dtypes:")
print(trainable_dtypes)

torch.cuda.empty_cache()

Trainable parameter dtypes:
{'torch.float32': 5144576}


In [9]:
# ============================================================
# 8. Helper functions for image-conditioned training
# ============================================================

def extract_tensor_from_image_encoding(encoded):
    """Moondream revisions can return a tensor, tuple, list, or dict. Normalize to tensor."""
    if torch.is_tensor(encoded):
        return encoded

    if isinstance(encoded, dict):
        for key in ["image_embeds", "embeds", "last_hidden_state", "hidden_states"]:
            if key in encoded and torch.is_tensor(encoded[key]):
                return encoded[key]
        for value in encoded.values():
            if torch.is_tensor(value):
                return value

    if isinstance(encoded, (tuple, list)):
        for value in encoded:
            if torch.is_tensor(value):
                return value

    raise TypeError(f"Cannot extract tensor from image encoding type: {type(encoded)}")


def get_text_embedding_layer():
    if hasattr(model.text_model, "get_input_embeddings"):
        return model.text_model.get_input_embeddings()
    if hasattr(model.text_model, "base_model") and hasattr(model.text_model.base_model, "get_input_embeddings"):
        return model.text_model.base_model.get_input_embeddings()
    raise AttributeError("Could not find input embedding layer for text_model.")


def encode_image_for_training(image):
    with torch.no_grad():
        encoded = model.encode_image(image)
        image_embeds = extract_tensor_from_image_encoding(encoded)

    if image_embeds.dim() == 2:
        image_embeds = image_embeds.unsqueeze(0)

    image_embeds = image_embeds.to(device=DEVICE, dtype=next(model.text_model.parameters()).dtype)
    return image_embeds.squeeze(0)  # [image_seq_len, hidden_dim]


def tokenize_text(text, add_special_tokens=False, max_length=None):
    return tokenizer(
        text,
        add_special_tokens=add_special_tokens,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )["input_ids"].squeeze(0)


def build_one_training_example(item):
    """
    Builds one sequence:
        [image embeddings] + [question/prompt token embeddings] + [answer token embeddings]

    Labels:
        -100 for image/question tokens
        answer token ids for answer tokens
    """
    image_embeds = encode_image_for_training(item["image"])

    question_text = f"Question: {item['question']}\nAnswer:"
    answer_text = item["answer"] + tokenizer.eos_token

    question_ids = tokenize_text(question_text, add_special_tokens=True, max_length=MAX_TEXT_TOKENS)
    answer_ids = tokenize_text(answer_text, add_special_tokens=False, max_length=48)

    question_ids = question_ids.to(DEVICE)
    answer_ids = answer_ids.to(DEVICE)

    embed_layer = get_text_embedding_layer()
    question_embeds = embed_layer(question_ids).to(dtype=image_embeds.dtype)
    answer_embeds = embed_layer(answer_ids).to(dtype=image_embeds.dtype)

    inputs_embeds = torch.cat([image_embeds, question_embeds, answer_embeds], dim=0)

    labels = torch.full(
        (inputs_embeds.shape[0],),
        -100,
        dtype=torch.long,
        device=DEVICE,
    )

    answer_start = image_embeds.shape[0] + question_embeds.shape[0]
    labels[answer_start:] = answer_ids

    attention_mask = torch.ones(inputs_embeds.shape[0], dtype=torch.long, device=DEVICE)

    return inputs_embeds, attention_mask, labels


def collate_fn(batch):
    examples = [build_one_training_example(item) for item in batch]

    max_len = max(x[0].shape[0] for x in examples)
    hidden = examples[0][0].shape[1]
    dtype = examples[0][0].dtype

    batch_inputs = torch.zeros((len(batch), max_len, hidden), dtype=dtype, device=DEVICE)
    batch_attention = torch.zeros((len(batch), max_len), dtype=torch.long, device=DEVICE)
    batch_labels = torch.full((len(batch), max_len), -100, dtype=torch.long, device=DEVICE)

    for i, (inputs_embeds, attention_mask, labels) in enumerate(examples):
        seq_len = inputs_embeds.shape[0]
        batch_inputs[i, :seq_len] = inputs_embeds
        batch_attention[i, :seq_len] = attention_mask
        batch_labels[i, :seq_len] = labels

    return {
        "inputs_embeds": batch_inputs,
        "attention_mask": batch_attention,
        "labels": batch_labels,
    }


print("Training helpers ready.")

Training helpers ready.


In [10]:
# ============================================================
# 9. Smoke test one batch before full training
# ============================================================

smoke_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=collate_fn,
)

batch = next(iter(smoke_loader))

print("inputs_embeds:", batch["inputs_embeds"].shape, batch["inputs_embeds"].dtype)
print("attention_mask:", batch["attention_mask"].shape)
print("labels:", batch["labels"].shape)
print("supervised tokens:", (batch["labels"] != -100).sum().item())

model.text_model.train()

with torch.cuda.amp.autocast(dtype=torch.float16):
    out = model.text_model(
        inputs_embeds=batch["inputs_embeds"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"],
    )

print("Smoke test loss:", float(out.loss.detach().cpu()))
assert torch.isfinite(out.loss), "Loss is not finite. Stop and inspect the batch."

inputs_embeds: torch.Size([1, 793, 2048]) torch.float16
attention_mask: torch.Size([1, 793])
labels: torch.Size([1, 793])
supervised tokens: 7


/tmp/ipykernel_1207/2029345047.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


Smoke test loss: 3.2212891578674316


In [11]:
# ============================================================
# 10. Training loop - OOM-safe version
# ============================================================

import gc
torch.cuda.empty_cache()
gc.collect()

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,   # should be 1
    shuffle=True,
    collate_fn=collate_fn,
)

trainable_params = [p for p in model.text_model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

scaler = torch.amp.GradScaler("cuda")

global_step = 0
loss_history = []

model.text_model.train()

for epoch in range(1, NUM_EPOCHS + 1):
    running = 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")

    for step, batch in enumerate(pbar, start=1):
        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = model.text_model(
                inputs_embeds=batch["inputs_embeds"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        running += loss.item() * GRAD_ACCUM_STEPS

        is_accum_step = step % GRAD_ACCUM_STEPS == 0
        is_last_step = step == len(train_loader)

        if is_accum_step or is_last_step:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                [p for p in model.text_model.parameters() if p.requires_grad],
                max_norm=1.0,
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            torch.cuda.empty_cache()

        avg_loss = running / step
        pbar.set_postfix({
            "loss": f"{avg_loss:.4f}",
            "step": global_step,
            "vram": f"{torch.cuda.memory_allocated() / 1e9:.2f}GB"
        })

    epoch_loss = running / len(train_loader)
    loss_history.append({"epoch": epoch, "loss": epoch_loss})

    epoch_dir = Path(CHECKPOINT_DIR) / f"epoch_{epoch}"
    epoch_dir.mkdir(parents=True, exist_ok=True)

    model.text_model.save_pretrained(str(epoch_dir))
    tokenizer.save_pretrained(str(epoch_dir))

    print(f"Epoch {epoch} complete. Loss: {epoch_loss:.4f}")
    print(f"Saved checkpoint to: {epoch_dir}")

print("Training done.")
pd.DataFrame(loss_history)

Epoch 1/3:   0%|          | 0/4000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:148: UserWarning: Could not find a config file in  - will assume that the vocabulary was not modified.
  warnings.warn(


Epoch 1 complete. Loss: 0.2461
Saved checkpoint to: /content/drive/MyDrive/ASTA/checkpoints/moondream2_lora/epoch_1


Epoch 2/3:   0%|          | 0/4000 [00:00<?, ?it/s]

Epoch 2 complete. Loss: 0.1620
Saved checkpoint to: /content/drive/MyDrive/ASTA/checkpoints/moondream2_lora/epoch_2


Epoch 3/3:   0%|          | 0/4000 [00:00<?, ?it/s]

Epoch 3 complete. Loss: 0.1494
Saved checkpoint to: /content/drive/MyDrive/ASTA/checkpoints/moondream2_lora/epoch_3
Training done.


,epoch,loss
0,1,0.246098
1,2,0.162047
2,3,0.149354


In [12]:
# ============================================================
# 11. Save final LoRA adapter
# ============================================================

Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

model.text_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved final adapter to:", SAVE_DIR)
print("Files:")
for p in Path(SAVE_DIR).iterdir():
    print(" -", p.name)

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:148: UserWarning: Could not find a config file in  - will assume that the vocabulary was not modified.
  warnings.warn(


Saved final adapter to: /content/drive/MyDrive/ASTA/moondream_season_lora
Files:
 - README.md
 - adapter_model.safetensors
 - adapter_config.json
 - tokenizer_config.json
 - special_tokens_map.json
 - added_tokens.json
 - vocab.json
 - merges.txt
 - tokenizer.json


In [13]:
# ============================================================
# 12. Prediction parsing helpers
# ============================================================

def normalize_word(value, valid_values):
    if value is None:
        return None
    value = str(value).strip().lower()
    for valid in valid_values:
        if value == valid.lower():
            return valid
    return None


def extract_json_object(text):
    if not text:
        return None
    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None


def parse_prediction(raw_text):
    obj = extract_json_object(raw_text)

    if isinstance(obj, dict):
        season = normalize_word(obj.get("season"), VALID_SEASONS)
        if season:
            return season

    lowered = str(raw_text).lower()

    for season in VALID_SEASONS:
        if season.lower() in lowered:
            return season

    return "PARSE_FAIL"


def predict_one(image_path, prompt=PROMPT):
    image = Image.open(image_path).convert("RGB")

    model.eval()
    model.text_model.eval()

    with torch.no_grad():
        enc_image = model.encode_image(image)

        try:
            raw = model.answer_question(enc_image, prompt, tokenizer)
        except Exception:
            raw = model.answer_question(image, prompt, tokenizer)

    pred = parse_prediction(raw)
    return raw, pred


print("Prediction helpers ready.")

Prediction helpers ready.


In [14]:
# ============================================================
# 13. Quick sanity check predictions
# ============================================================

sample_indices = random.sample(range(len(test_ds)), k=min(12, len(test_ds)))
print(PROMPT)
for idx in sample_indices:
    sample = test_ds.samples[idx]
    raw, pred = predict_one(sample["image_path"])

    print("-" * 80)
    print("Image:", sample["relative_image_path"])
    print("Expected:", sample["label"])
    print("Raw:", raw)
    print("Parsed:", pred)

Classify this face into exactly one of these 4 seasonal color classes:

Autumn, Spring, Summer, Winter.

Return ONLY this JSON format:
{"season":"Winter"}

No explanation. No markdown. No extra words.


--------------------------------------------------------------------------------
Image: data/training_crops/test/spring/test_spring_3639.jpg
Expected: Spring
Raw: {"season":"Summer"}
Parsed: Summer
--------------------------------------------------------------------------------
Image: data/training_crops/test/autumn/test_autumn_27382.jpg
Expected: Autumn
Raw: {"season":"Autumn"}
Parsed: Autumn
--------------------------------------------------------------------------------
Image: data/training_crops/test/summer/test_summer_27236.jpg
Expected: Summer
Raw: {"season":"Summer"}
Parsed: Summer
--------------------------------------------------------------------------------
Image: data/training_crops/test/autumn/test_autumn_19413.jpg
Expected: Autumn
Raw: {"season":"Autumn"}
Parsed: Autumn
--------------------------------------------------------------------------------
Image: data/training_crops/test/winter/test_winter_3706.jpg
Expected: Winter
Raw: {"season":"Winter"}
Parsed: Winter
-------

In [15]:
# ============================================================
# 14. Full test evaluation - 4-season version
# ============================================================

y_true = []
y_pred = []
rows = []

for sample in tqdm(test_ds.samples, desc="Evaluating test set"):
    raw, pred = predict_one(sample["image_path"])

    expected = sample["label"]

    y_true.append(expected)
    y_pred.append(pred)

    rows.append({
        "image_path": sample["relative_image_path"],
        "expected": expected,
        "prediction": pred,
        "raw_output": raw,
        "correct": expected == pred,
        "parse_ok": pred != "PARSE_FAIL",
    })

results_df = pd.DataFrame(rows)

json_validity_rate = results_df["parse_ok"].mean()
season_accuracy = results_df["correct"].mean()

print("JSON / parse validity rate:", round(json_validity_rate * 100, 2), "%")
print("Season accuracy:", round(season_accuracy * 100, 2), "%")

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=CLASS_NAMES,
    zero_division=0
))

results_path = f"{PROJECT_ROOT}/moondream2_test_predictions.csv"
results_df.to_csv(results_path, index=False)

print("\nSaved predictions to:", results_path)

Evaluating test set:   0%|          | 0/912 [00:00<?, ?it/s]

JSON / parse validity rate: 100.0 %
Season accuracy: 51.54 %

Classification report:
              precision    recall  f1-score   support

      Autumn       0.60      0.30      0.40       259
      Spring       0.47      0.17      0.25       203
      Summer       0.40      0.83      0.54       186
      Winter       0.62      0.78      0.69       264

    accuracy                           0.52       912
   macro avg       0.52      0.52      0.47       912
weighted avg       0.54      0.52      0.48       912


Saved predictions to: /content/drive/MyDrive/ASTA/moondream2_test_predictions.csv


In [16]:
# ============================================================
# 15. Confusion matrix
# ============================================================

labels_for_cm = CLASS_NAMES + (["PARSE_FAIL"] if "PARSE_FAIL" in set(y_pred) else [])

cm = confusion_matrix(y_true, y_pred, labels=labels_for_cm)

cm_df = pd.DataFrame(cm, index=labels_for_cm, columns=labels_for_cm)
cm_path = f"{PROJECT_ROOT}/moondream2_confusion_matrix.csv"
cm_df.to_csv(cm_path)

display(cm_df)

print("Saved confusion matrix to:", cm_path)

,Autumn,Spring,Summer,Winter
Autumn,77,26,65,91
Spring,18,34,137,14
Summer,7,6,154,19
Winter,26,6,27,205


Saved confusion matrix to: /content/drive/MyDrive/ASTA/moondream2_confusion_matrix.csv


## After Training

Download/copy this final adapter folder from Drive into your local project:

```text
MyDrive/ASTA/moondream_season_lora
```

Place it in your app root as:

```text
Skin-Color-Analysis/
├── analyzer.py
├── main.py
└── moondream_season_lora/
```

Then run locally:

```powershell
streamlit run main.py
```

For the report, record:

- JSON / parse validity rate
- Season accuracy
- Subtype accuracy
- Full class accuracy
- Per-class classification report
- Confusion matrix